In [ ]:
# Copyright (c) 2026 Constructor Technology AG
#
# This file is part of cu_cilia, released under the GNU General Public
# License v3.0. See the LICENSE file for details.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from src.cilia_detection.metrics import segmentation_metrics, detection_metrics, calc_precision_recall
from src.cilia_detection.utils import yolobbox2bbox
import cv2

In [2]:
TYPE = "easy"
proj_dir = Path(f"../../data/cilia_dataset/{TYPE}")

# Detection metrics

In [3]:
gt_dir = proj_dir / "cellprofiler_results"
pred_dir = proj_dir / "cilia_results/bboxes"

In [4]:
img_results = {}
for p in gt_dir.glob("*.txt"):
    with open(p, 'r') as f:
        lines = f.readlines()
        gt_bboxes = [yolobbox2bbox([float(p) for p in line.split(' ')[1:]]) for line in lines]
    with open(pred_dir / p.name, 'r') as f:
        lines = f.readlines()
        pred_bboxes = [yolobbox2bbox([float(p) for p in line.split(' ')[1:]]) for line in lines]

    img_results[p.stem] = detection_metrics(gt_bboxes, pred_bboxes, 0.5)
p, r, f1 = calc_precision_recall(img_results)
print('Easy cases:')
print('\tPrecision:', p, 'Recall:', r, 'F1-score:', f1)

Easy cases:
	Precision: 0.9122807017543859 Recall: 0.7647058823529411 F1-score: 0.8319995038722958


In [42]:
metrics = pd.DataFrame(img_results).transpose().sort_index().round(3)
metrics.to_csv(pred_dir.parent / "detection_metrics.csv")
metrics

,true_pos,false_pos,false_neg,accuracy,precision,recall,f1
Nthy-oriZ_untreated_10,8.0,1.0,0.0,0.889,0.889,1.000,0.941
Nthy-oriZ_untreated_2,7.0,1.0,2.0,0.700,0.875,0.778,0.824
Nthy-oriZ_untreated_3,11.0,0.0,6.0,0.647,1.000,0.647,0.786
Nthy-oriZ_untreated_4,6.0,2.0,6.0,0.429,0.750,0.500,0.600
Nthy-oriZ_untreated_6,20.0,1.0,2.0,0.870,0.952,0.909,0.930


# Segmentation metrics

In [36]:
gt_dir = proj_dir / "cellprofiler_results"
pred_dir = proj_dir / "cilia_results/labels"

img_results = {}
for p in gt_dir.glob("*.npy"):
    gt_mask = np.load(p).astype(np.uint8) * 255
    pred_mask = cv2.imread((pred_dir / p.name.replace("_binary", "")).with_suffix(".png").as_posix(), cv2.IMREAD_GRAYSCALE)
    img_results[p.stem] = segmentation_metrics(gt_mask, pred_mask)

In [37]:
seg_metrics = pd.DataFrame(img_results).transpose().sort_index().round(3)
seg_metrics.to_csv(pred_dir.parent / "segmentation_metrics.csv")
seg_metrics

,accuracy,precision,recall,f1,iou,boundary_f1,hausdorff_distance,specificity,fpr
Nthy-oriZ_untreated_10_binary,1.000,0.691,0.887,0.777,0.636,0.117,455.874,1.000,0.000
Nthy-oriZ_untreated_2_binary,1.000,0.725,0.922,0.812,0.683,0.092,430.643,1.000,0.000
Nthy-oriZ_untreated_3_binary,0.999,0.802,0.787,0.794,0.659,0.106,554.707,1.000,0.000
Nthy-oriZ_untreated_4_binary,1.000,0.692,0.670,0.681,0.516,0.071,956.866,1.000,0.000
Nthy-oriZ_untreated_6_binary,0.999,0.556,0.951,0.702,0.541,0.060,585.284,0.999,0.001
